<a href="https://colab.research.google.com/github/Reginajose/Observatorio-Clima-Grande-desafio/blob/main/transformacao_filtros.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Transformação dos Dados e Análises Estatísticas**



IMPORTANTO AS BIBILHOTECAS

In [ ]:
import pandas as pd
import numpy as np

# Leitura do Dataset Mestre
df_mestre = pd.read_csv(
    'dataset_mestre_observatorio.csv',
    dtype={'Cod_IBGE_7': str},
    encoding='utf-8-sig'
)

print(f"Dataset Mestre carregado: {df_mestre.shape[0]} municípios e {df_mestre.shape[1]} colunas.")

Dataset Mestre carregado: 5570 municípios e 15 colunas.


**1 - Aplicação dos Filtros do Funil**

In [ ]:


# 1. Base Inicial
f0 = df_mestre.copy()

# 2. Filtro 1: Participou do SINISA
f1 = f0[
    (f0['participou_sinisa'] == True)
    | (
        f0['participou_sinisa']
        .astype(str)
        .str.strip()
        .str.upper()
        .isin(['TRUE', 'SIM', '1'])
    )
].copy()

# 3. Filtro 2: Domicílios em área de risco (> 0)
f1['qtd_domicilios_risco'] = pd.to_numeric(
    f1['qtd_domicilios_risco'], errors='coerce'
).fillna(0)
f1['pct_domicilios_risco'] = pd.to_numeric(
    f1['pct_domicilios_risco'], errors='coerce'
).fillna(0)
f2 = f1[f1['qtd_domicilios_risco'] > 0].copy()

# 4. Filtro 3: Fragilidade Fiscal / Sem Acesso a Crédito Bancário
# ATUALIZADO: Exclui apenas A, A+, B, B+ (Mantém C, D, n.d., n.e. e Nulos)
f2['capag_limpa'] = (
    f2['capag'].fillna('').astype(str).str.strip().str.upper()
)
f3 = f2[~f2['capag_limpa'].isin(['A', 'A+', 'B', 'B+'])].copy()

# 5. Filtro 4: Mapeamento de Risco Confirmado
f4 = f3[
    f3['mapeamento_area_risco']
    .fillna('')
    .astype(str)
    .str.strip()
    .str.upper()
    .isin(['SIM', 'S', '1', 'TRUE'])
].copy()

print('\n=== MÉTRICA DO FUNIL ATUALIZADA ===')
print(
    f'Total Inicial: {len(f0)} | SINISA: {len(f1)} | Com Risco: {len(f2)} |'
    f' CAPAG C/D/Sem Nota: {len(f3)} | Elegíveis Finais: {len(f4)}'
)


=== MÉTRICA DO FUNIL ATUALIZADA ===
Total Inicial: 5570 | SINISA: 4845 | Com Risco: 2238 | CAPAG C/D/Sem Nota: 1077 | Elegíveis Finais: 641


**Ordenação e Construção do Raking - TOP 20** -
(OPÇÃO 70% QTD / 30% PCT)

In [ ]:
f4['rank_qtd_pctile'] = f4['qtd_domicilios_risco'].rank(pct=True)
f4['rank_pct_pctile'] = f4['pct_domicilios_risco'].rank(pct=True)
f4['nota_70_30'] = (0.7 * f4['rank_qtd_pctile']) + (0.3 * f4['rank_pct_pctile'])

top_20 = f4.nlargest(20, 'nota_70_30').copy().reset_index(drop=True)
top_20['posicao'] = top_20.index + 1

In [ ]:
# Adicionando uma coluna de posição utilizando a variável correta (top_20).
top_20['posicao'] = top_20.index + 1

top_20.head(20)

,Cod_IBGE_7,Nome,UF,mapeamento_area_risco,qtd_domicilios_risco,municipio_critico,pct_vias_rede_drenagem,investimento_percapita_drenagem,pct_domicilios_risco,participou_sinisa,capag,criterio_capag_cd,total_desastres,soma_danos_materiais,soma_afetados,capag_limpa,rank_qtd_pctile,rank_pct_pctile,nota_70_30,posicao
0,4307708,Esteio,RS,Sim,30000.0,Sim,98.63,0.00,88.18,True,C,True,5,20477184.43,20814.0,C,0.992200,0.998440,0.994072,1
1,4304606,Canoas,RS,Sim,70000.0,Sim,96.81,166.99,43.53,True,C,True,7,6910085.69,184308.0,C,0.996880,0.981279,0.992200,2
2,3304904,São Gonçalo,RJ,Sim,175838.0,Sim,5.60,39.73,40.00,True,C,True,4,27405589.64,605.0,C,1.000000,0.968799,0.990640,3
3,5002704,Campo Grande,MS,Sim,125000.0,Sim,40.84,7.70,30.43,True,C,True,56,112000.00,0.0,C,0.998440,0.940718,0.981123,4
4,3303708,Paraíba do Sul,RJ,Sim,8561.0,Sim,45.45,0.00,46.47,True,C,True,3,3706249.92,1015.0,C,0.971919,0.989080,0.977067,5
5,3301009,Campos dos Goytacazes,RJ,Sim,63000.0,Sim,9.37,32.59,27.21,True,C,True,8,26239841.06,1142.0,C,0.995320,0.928237,0.975195,6
6,2601409,Barreiros,PE,Sim,7938.0,Sim,6.78,0.00,50.00,True,C,True,6,667387.98,7193.0,C,0.965679,0.995320,0.974571,7
7,2610004,Palmares,PE,Sim,8000.0,Sim,NaN,NaN,40.49,True,C,True,5,15709645.23,963.0,C,0.968019,0.971919,0.969189,8
8,1600279,Laranjal do Jari,AP,Sim,6000.0,Sim,18.90,0.00,45.01,True,n.d.,False,2,81766.53,388.0,N.D.,0.959438,0.987520,0.967863,9
9,3113305,Carangola,MG,Sim,6000.0,Sim,81.97,26.18,43.52,True,C,True,2,16970153.71,5502.0,C,0.959438,0.979719,0.965523,10


DISTRIBUIÇÃO DA VERBA - OPÇÃO B (PROPORCIONAL TAMANHO DO MUNICÍPIO)

In [ ]:
VERBA_TOTAL = 150_000_000  # R$ 150 Milhões
PISO_MINIMO = 3_000_000  # R$ 3 Milhões (Piso)
TETO_MAXIMO = 15_000_000  # R$ 15 Milhões (Teto)

soma_moradias_top20 = top_20['qtd_domicilios_risco'].sum()

# Cálculo proporcional inicial
top_20['verba_bruta'] = (
    top_20['qtd_domicilios_risco'] / soma_moradias_top20
) * VERBA_TOTAL

# Aplicação do Piso e Teto
top_20['verba_ajustada'] = top_20['verba_bruta'].clip(
    lower=PISO_MINIMO, upper=TETO_MAXIMO
)

# Ajuste fino para fechar em exatamente R$ 150 Milhões
diferenca = VERBA_TOTAL - top_20['verba_ajustada'].sum()
mascara_livre = (top_20['verba_ajustada'] > PISO_MINIMO) & (
    top_20['verba_ajustada'] < TETO_MAXIMO
)

if mascara_livre.sum() > 0:
  soma_livres = top_20.loc[mascara_livre, 'qtd_domicilios_risco'].sum()
  top_20.loc[mascara_livre, 'verba_ajustada'] += (
      top_20.loc[mascara_livre, 'qtd_domicilios_risco'] / soma_livres
  ) * diferenca

top_20['verba_alocada_m'] = (top_20['verba_ajustada'] / 1_000_000).round(2)

# Mapeamento do perfil fiscal
top_20['situacao_fiscal'] = (
    top_20['capag_limpa']
    .map({
        'C': 'Fragilidade Fiscal',
        'D': 'Alta Fragilidade Fiscal',
        'N.D.': 'Pendência Contábil (Sem Nota)',
        'N.E.': 'Não Elegível (Sem Nota)',
    })
    .fillna('Sem Nota/Pendente')
)

**Corte e cálculo "custo de não agir"**

In [ ]:
# Métrica de cobertura do orçamento
demais_elegiveis = f4[~f4['Cod_IBGE_7'].isin(top_20['Cod_IBGE_7'])].copy()

# Garantir conversão numérica das colunas do Atlas de Desastres
cols_atlas = ['total_desastres', 'soma_danos_materiais', 'soma_afetados']
for col in cols_atlas:
  top_20[col] = pd.to_numeric(top_20[col], errors='coerce').fillna(0)
  demais_elegiveis[col] = (
      pd.to_numeric(demais_elegiveis[col], errors='coerce').fillna(0)
  )

tabela_custo_nao_agir = pd.DataFrame([
    {
        'Grupo': '20 Selecionados (Top 20)',
        'Qtd Cidades': len(top_20),
        'Moradias em Risco (SINISA)': (
            f"{top_20['qtd_domicilios_risco'].sum():,.0f}".replace(',', '.')
        ),
        'Total de Desastres (2021-2025)': (
            f"{top_20['total_desastres'].sum():,.0f}".replace(',', '.')
        ),
        'Danos Materiais Totais (R$)': (
            f"R$ {top_20['soma_danos_materiais'].sum():,.2f}".replace(
                ',', 'X'
            )
            .replace('.', ',')
            .replace('X', '.')
        ),
        'Pessoas Afetadas (Danos Humanos)': (
            f"{top_20['soma_afetados'].sum():,.0f}".replace(',', '.')
        ),
    },
    {
        'Grupo': 'Demais Elegíveis (Fora do Top 20)',
        'Qtd Cidades': len(demais_elegiveis),
        'Moradias em Risco (SINISA)': (
            f"{demais_elegiveis['qtd_domicilios_risco'].sum():,.0f}".replace(
                ',', '.'
            )
        ),
        'Total de Desastres (2021-2025)': (
            f"{demais_elegiveis['total_desastres'].sum():,.0f}".replace(
                ',', '.'
            )
        ),
        'Danos Materiais Totais (R$)': (
            f"R$ {demais_elegiveis['soma_danos_materiais'].sum():,.2f}".replace(
                ',', 'X'
            )
            .replace('.', ',')
            .replace('X', '.')
        ),
        'Pessoas Afetadas (Danos Humanos)': (
            f"{demais_elegiveis['soma_afetados'].sum():,.0f}".replace(',', '.')
        ),
    },
])

print('=== TABELA: CUSTO DE NÃO AGIR ===')
display(tabela_custo_nao_agir)

=== TABELA: CUSTO DE NÃO AGIR ===


,Grupo,Qtd Cidades,Moradias em Risco (SINISA),Total de Desastres (2021-2025),Danos Materiais Totais (R$),Pessoas Afetadas (Danos Humanos)
0,20 Selecionados (Top 20),20,598.353,164,"R$ 628.128.636,39",275.775
1,Demais Elegíveis (Fora do Top 20),621,525.525,1.649,"R$ 9.202.919.613,21",792.960


Texte de Consistências

In [ ]:
moradias_atendidas = top_20['qtd_domicilios_risco'].sum()
moradias_totais_elegiveis = f4['qtd_domicilios_risco'].sum()
pct_cobertura = (moradias_atendidas / moradias_totais_elegiveis) * 100

print('\n=== TESTES DE CONSISTÊNCIA E VALIDAÇÃO DE REGRAS ===')

# Teste 1: Validação do Funil e Mapeamento
mapeamento_ok = (top_20['mapeamento_area_risco'] == 'Sim').sum()
print(
    f'✅ Teste 1 (Mapeamento de Risco): {mapeamento_ok} de 20 municípios no'
    ' Top 20 possuem Mapeamento de Risco confirmado.'
)

# Teste 2: Validação da Regra CAPAG (Exclusão de A, A+, B, B+)
capag_invalidas = top_20['capag_limpa'].isin(['A', 'A+', 'B', 'B+']).sum()
if capag_invalidas == 0:
  print(
      '✅ Teste 2 (Filtro Fiscal): 100% dos municípios no Top 20 possuem'
      ' restrição de acesso a crédito (CAPAG C, D ou Sem Nota).'
  )

# Teste 3: Cobertura de Moradias no Orçamento
print(
    f'✅ Teste 3 (Cobertura de Risco): O Top 20 protege'
    f' {moradias_atendidas:,.0f} moradias de um total de'
    f' {moradias_totais_elegiveis:,.0f} nos elegíveis ({pct_cobertura:.1f}% de'
    ' cobertura).'
)

# Teste 4: Fechamento Orçamentário (Verba = R$ 150 Mi)
verba_fechamento = top_20['verba_ajustada'].sum()
print(
    f'✅ Teste 4 (Fechamento Orçamentário): Soma total alocada no Top 20 = R$'
    f' {verba_fechamento/1e6:.2f} Mi (Meta: R$ 150,00 Mi).'
)


=== TESTES DE CONSISTÊNCIA E VALIDAÇÃO DE REGRAS ===
✅ Teste 1 (Mapeamento de Risco): 20 de 20 municípios no Top 20 possuem Mapeamento de Risco confirmado.
✅ Teste 2 (Filtro Fiscal): 100% dos municípios no Top 20 possuem restrição de acesso a crédito (CAPAG C, D ou Sem Nota).
✅ Teste 3 (Cobertura de Risco): O Top 20 protege 598,353 moradias de um total de 1,123,878 nos elegíveis (53.2% de cobertura).
✅ Teste 4 (Fechamento Orçamentário): Soma total alocada no Top 20 = R$ 150.00 Mi (Meta: R$ 150,00 Mi).


EXPORTAÇÃO DOS ARQUIVOS PARA O LOOKER STUDIO

In [ ]:
colunas_export = [
    'posicao',
    'Cod_IBGE_7',
    'Nome',
    'UF',
    'capag',
    'situacao_fiscal',
    'qtd_domicilios_risco',
    'verba_alocada_m',
    'total_desastres',
    'soma_afetados',
    'soma_danos_materiais',
]

tabela_export = top_20[colunas_export].copy()

# Exportação para CSV e Excel
csv_nome = 'Painel_v1_Top20_OpcaoB_ComSemNota.csv'
xlsx_nome = 'Painel_v1_Top20_OpcaoB_ComSemNota.xlsx'

tabela_export.to_csv(csv_nome, index=False, encoding='utf-8-sig')
tabela_export.to_excel(xlsx_nome, index=False)

print(
    f"\n🚀 Sucesso! Arquivos '{csv_nome}' e '{xlsx_nome}' prontos para o Looker"
    ' Studio.'
)

files.download(csv_nome)
files.download(xlsx_nome)


🚀 Sucesso! Arquivos 'Painel_v1_Top20_OpcaoB_ComSemNota.csv' e 'Painel_v1_Top20_OpcaoB_ComSemNota.xlsx' prontos para o Looker Studio.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>